# Navegar e verificar lineage

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
from src.workbench.manual_validation import display_locations, audit, export_session
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="08_auditoria_lineage"


{'mode': 'READ_EXISTING',
 'proposal_id': 'ca4539df-4a11-4647-9d77-2cc36a484968',
 'snapshot_id': None,
 'model_name': 'planilha-turmas-demo',
 'model_version': '',
 'registry_prefix': '',
 'registry_page_token': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': False,
 'confirmation': '',
 'request_key': '',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 3,
 'campaign_display_name': 'POC DEMO • Validação manual',
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': None}

{'state': 'READY',
 'write_enabled': False,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Escolha proposta, run, modelo+versão, snapshot ou campanha; paginação limita o grafo retornado.

In [2]:
from src.workbench.lineage_review import inspect_graph
external_id=params.external_id or (str(params.proposal_id) if params.proposal_id else "")
if not external_id: raise ValueError("Informe external_id ou proposal_id")
evidence=inspect_graph(cfg,params.lineage_kind,external_id,params.model_version)
display(pd.DataFrame(evidence["nodes"]));display(pd.DataFrame(evidence["edges"]))


,entity_id,entity_type,external_id,environment,domain,created_at_utc,attributes
0,f2088b18-0a06-4413-872b-3573b404aa23,campaign,28d5c5e0-e21d-458c-8e0a-3aca7b4acc31,poc,planilha-turmas,2026-11-04T18:58:35.755561+00:00,{}
1,bd8c45c2-c137-45e5-92d1-d89be5b1f94b,campaign,39f8c7aa-0e6a-463c-aa77-3748f6d118d5,poc,planilha-turmas,2026-11-04T20:12:06.733851+00:00,{}
2,91c5f4d2-d9d8-409f-9b85-c532f5a649c2,campaign,4626d239-6145-4cc3-95ca-5640799dff03,poc,planilha-turmas,2026-11-04T21:48:09.751781+00:00,{}
3,84922b87-59fb-48aa-9935-5be737802ae1,campaign,4e413355-a21c-4c13-9311-0320857de96c,poc,planilha-turmas,2026-11-04T18:13:16.517767+00:00,{}
4,bd1aab5a-9467-4ed8-a71e-d822ea1cd00c,campaign,59f2d8ec-65fc-44ae-aa94-46f82adaf2a7,poc,planilha-turmas,2026-11-04T21:31:37.386305+00:00,{}
...,...,...,...,...,...,...,...
95,9fa0dcae-ce48-4e4b-bb1f-77a6a551d0b2,run,a4066f48ab8542d28660d03a5a06e578,poc,planilha-turmas,2026-11-04T21:48:09.751781+00:00,{}
96,e615b00c-2317-48f3-abce-0191b18e5f98,run,aafa93b0e17d4ed8830e629bc178d13e,poc,planilha-turmas,2026-11-04T18:13:28.219589+00:00,{}
97,caf444ee-af21-46e2-b177-065fab496887,run,c0d9f3138c874049b339e0f54ab8348c,poc,planilha-turmas,2026-11-04T19:24:24.629371+00:00,{}
98,dde8f698-7130-47cf-b1a3-f653b683667a,run,c78c2971d2d246efb3fcacc5cdf4cd71,poc,planilha-turmas,2026-11-04T19:24:42.416011+00:00,{}


,edge_id,parent_id,child_id,relation_type,created_at_utc
0,004fa837-0977-4157-a971-c10699e4c30f,9a30932f-0885-48b8-9ecf-f07385436ddf,ff634d3b-f5a7-4e46-b140-2f6abe9b38bc,REGISTERED_AS,2026-11-04T21:31:38.214169+00:00
1,042f4a44-63da-4576-a04f-50f8d65ef159,70ae9d3c-2b48-4bd4-a565-91d18ec39d42,f52962cd-0110-4425-ae7f-1e28570bf565,TRAINED_ON,2026-11-04T20:12:06.733851+00:00
2,05e4f446-57da-4f6d-be05-d6d961d2fe51,c1f4b77e-a4f5-4e47-a253-15dfb692e9ac,39bae136-ab87-4000-8d2c-4466293d336f,CONTRIBUTED_TO,2026-11-04T21:48:20.494736+00:00
3,0692e507-1e2d-41cd-9c0a-8867c02c8268,cb4c117b-518f-4b12-ba3d-dca777b24e19,7909d968-4411-458e-88b6-e60c51e84152,REGISTERED_AS,2026-11-04T21:31:37.392227+00:00
4,09c1d6b6-f9ef-4540-82b9-30994d352229,f3a5ced0-1bf7-412f-9fac-7e62144b8f89,cb18b99d-bb8d-4242-8f7c-9eddbdc84fbe,PREDICTED_WITH,2026-11-04T21:31:43.053407+00:00
...,...,...,...,...,...
95,abac8b58-d662-4248-b36b-21ce39b2a635,91c5f4d2-d9d8-409f-9b85-c532f5a649c2,9fa0dcae-ce48-4e4b-bb1f-77a6a551d0b2,CONTRIBUTED_TO,2026-11-04T21:48:09.751781+00:00
96,abf66d18-7aa9-495d-bdb7-b2b93bcb7337,4b192779-be1a-46b7-9b9c-d8a1b3f20c35,a8619814-7b05-4786-82b9-0f914d7b0048,PREDICTED_WITH,2026-11-04T19:24:24.629371+00:00
97,ac25f01c-ce6c-4020-8f7b-f331327f2710,70ae9d3c-2b48-4bd4-a565-91d18ec39d42,ee774a99-3a2d-4bdd-8a08-3edcf137e890,TRAINED_ON,2026-11-04T18:13:16.517767+00:00
98,aee0778c-b50d-4698-baec-acb662629c6b,ff634d3b-f5a7-4e46-b140-2f6abe9b38bc,d7a7bc60-e33e-40f7-ac05-f7029c2858a7,CONTRIBUTED_TO,2026-11-04T21:31:43.053407+00:00


2. Grafo e verificações reais. VERIFIED exige presença/proveniência; MISSING/MISMATCH bloqueiam confiança; legado continua identificado.

In [3]:
from IPython.display import HTML
from src.workbench.lineage_review import render_graph
display(HTML(render_graph(evidence)))
display(pd.DataFrame(evidence["checks"]))
reconciled=audit(cfg,params.lineage_kind,external_id,params.model_version)
display(reconciled)
inventory_rows=call(cfg,"GET","/v1/audit/inventory?limit=500")
display(pd.DataFrame(inventory_rows))
display(export_session(reconciled,inventory_rows))


run: 870324ce4ccd4aee9bcfab4f4084c509,→ REGISTERED_AS →,model: hi.pt.demanda.DEMO-C1.DEMO-CR1.P2/1
70ae9d3c-2b48-4bd4-a565-91d18ec39d42,→ TRAINED_ON →,f52962cd-0110-4425-ae7f-1e28570bf565
campaign: cacd50e2-831d-4f11-a261-8485b011dc30,→ CONTRIBUTED_TO →,run: 0923144dd42149db841654c62c600c17
cb4c117b-518f-4b12-ba3d-dca777b24e19,→ REGISTERED_AS →,model: hi.pt.demanda.DEMO-C1.DEMO-CR2.P1/1
model: hi.pt.demanda.DEMO-C1.DEMO-CR1.P1/2,→ PREDICTED_WITH →,inference: 298eeb8a-bbb5-4edc-9f27-1076252bf96c
inference: 41ee90fe-0f08-48bf-a47a-6503edeac8c7,→ CONTRIBUTED_TO →,proposal: 8ba35d3e-70ee-4719-943a-4b607d2549ee
model: hi.demo.integration.d0b9bf81/1,→ PREDICTED_WITH →,inference: d987f81e-271d-4d0a-90de-4d5eec6812e9
model: planilha-turmas-demo/14,→ PREDICTED_WITH →,inference: 0f45a3cf-0864-4c07-85ac-56280f1882a2
campaign: 28d5c5e0-e21d-458c-8e0a-3aca7b4acc31,→ CONTRIBUTED_TO →,run: 9db2d218bc3845e1841f2f7a21f624a5
inference: 08e86401-59f1-421d-87a5-c43b676652e9,→ CONTRIBUTED_TO →,proposal: b6b8b14b-33a1-4de1-bc01-ca839afdde2d
campaign: 7e14b64f-6c41-4a7b-b3e7-33b5750afd1f,→ CONTRIBUTED_TO →,run: 18cdc4f10eaf47e4ab30104994e34ca5


,kind,id,status,evidence,url
0,campaign,28d5c5e0-e21d-458c-8e0a-3aca7b4acc31,VERIFIED,PostgreSQL lineage entity,NaN
1,campaign,39f8c7aa-0e6a-463c-aa77-3748f6d118d5,VERIFIED,PostgreSQL lineage entity,NaN
2,campaign,4626d239-6145-4cc3-95ca-5640799dff03,VERIFIED,PostgreSQL lineage entity,NaN
3,campaign,4e413355-a21c-4c13-9311-0320857de96c,VERIFIED,PostgreSQL lineage entity,NaN
4,campaign,59f2d8ec-65fc-44ae-aa94-46f82adaf2a7,VERIFIED,PostgreSQL lineage entity,NaN
...,...,...,...,...,...
95,run,a4066f48ab8542d28660d03a5a06e578,VERIFIED,PostgreSQL lineage entity,http://127.0.0.1:5000/#/experiments/5/runs/a40...
96,run,aafa93b0e17d4ed8830e629bc178d13e,VERIFIED,PostgreSQL lineage entity,http://127.0.0.1:5000/#/experiments/4/runs/aaf...
97,run,c0d9f3138c874049b339e0f54ab8348c,VERIFIED,PostgreSQL lineage entity,http://127.0.0.1:5000/#/experiments/4/runs/c0d...
98,run,c78c2971d2d246efb3fcacc5cdf4cd71,VERIFIED,PostgreSQL lineage entity,http://127.0.0.1:5000/#/experiments/4/runs/c78...


{'status': 'VERIFIED',
 'proposal_id': 'ca4539df-4a11-4647-9d77-2cc36a484968',
 'proposal_display_name': 'Planilha Turmas DEMO • DRAFT',
 'stored_rows': 9,
 'declared_rows': 9,
 'models': [{'model_key': 'hi.pt.demanda.DEMO-C1.DEMO-CR1.P1',
   'model_version': '4',
   'status': 'VERIFIED',
   'registry_accessible': True,
   'registered': True,
   'sql_approved': False,
   'approved': False,
   'artifact_verifiable': True,
   'checks': ['native_ids',
    'shared_tags',
    'finished_run',
    'signature',
    'proxy_equals_s3',
    'catalog_equals_manifest',
    'snapshot_sql_equals_manifest'],
   'serving': {'healthy': True,
    'cached': True,
    'cache': {'hits': 0, 'misses': 4, 'maxsize': 4, 'currsize': 4}},
   'run_id': '18cdc4f10eaf47e4ab30104994e34ca5',
   'artifact_uri': 'models:/m-580070f5f05c467f8c3fb922b69d9913',
   'aliases': {},
   'metadata': {'files': [{'key': 'mlflow/5/models/m-580070f5f05c467f8c3fb922b69d9913/artifacts/MLmodel',
      'bytes': 1336,
      'bucket': 'puc

,model_key,model_version,run_id,snapshot_id,campaign_id,model_display_name,cycle_code,campus_code,course_code,period_code,...,run_name,artifact_uri,artifact_sha256,dataset_sha256,algorithm,manifest_uri,manifest_sha256,integrity_status,sql_approved,metadata
0,hi.demo.integration.b8cf3663,1,f1a35ee7490045329e7c999207ca0f5e,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,b0a2f2ba-893f-4046-b378-757e46a27f76,Ensaio DEMO agregado • hi.demo.integration.b8c...,2026-1,None,None,None,...,hi.demo.integration.b8cf3663__ridge__20261104T...,models:/m-ca203faf8a6c4b01a9b44853e8faa2d4,4268666504b258b661b9c6cc009bdf8c21d06fde4317fb...,5fc39eeeead59aa48813182e6a93d2308a4a9243655501...,ridge,s3://pucpr-poc/lineage/poc/planilha-turmas/cat...,8aa6c09103b0ad9aac50a632c9a43b7a6f979bb8923f44...,RECORDED,False,{'files': [{'key': 'mlflow/5/models/m-ca203faf...
1,hi.demo.integration.be61af4a,1,0923144dd42149db841654c62c600c17,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,cacd50e2-831d-4f11-a261-8485b011dc30,Ensaio DEMO agregado • hi.demo.integration.be6...,2026-1,None,None,None,...,hi.demo.integration.be61af4a__ridge__20261104T...,models:/m-0af41fa8fe13489bb0b39f1e255f0f87,4268666504b258b661b9c6cc009bdf8c21d06fde4317fb...,5fc39eeeead59aa48813182e6a93d2308a4a9243655501...,ridge,s3://pucpr-poc/lineage/poc/planilha-turmas/cat...,e233f43208feecabd2e31daa2ca82e09f5390a062689e2...,RECORDED,False,{'files': [{'key': 'mlflow/5/models/m-0af41fa8...
2,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,5,2836a33c2571431da2bd061d413f1401,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,f76168f8-6eef-4fa9-9e5f-ccab2c0ab9ae,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR1 • Pe...,2026-1,DEMO-C1,DEMO-CR1,P1,...,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1__ridge__2026...,models:/m-d20bb98c195f4c2aaa841b6d40c6d6d5,ed906b992cc3d85b0641488fa01758037fd33efd7bd6f3...,22b3c1491ef5f01f1c646bc43fd8350d324bce87b3bf4e...,ridge,s3://pucpr-poc/lineage/poc/planilha-turmas/cat...,cbc924534922f1a479907ccccec3308ea755d690c3e88d...,RECORDED,False,{'files': [{'key': 'mlflow/5/models/m-d20bb98c...
3,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,4,18cdc4f10eaf47e4ab30104994e34ca5,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,7e14b64f-6c41-4a7b-b3e7-33b5750afd1f,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR1 • Pe...,2026-1,DEMO-C1,DEMO-CR1,P1,...,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1__ridge__2026...,models:/m-580070f5f05c467f8c3fb922b69d9913,8e6043209a63edaadbdd46de27731ca0c7fbcf9555c8d9...,22b3c1491ef5f01f1c646bc43fd8350d324bce87b3bf4e...,ridge,s3://pucpr-poc/lineage/poc/planilha-turmas/cat...,fc70250322f4b8336efc5992881c79080f530fb42ce1ca...,RECORDED,False,{'files': [{'key': 'mlflow/5/models/m-580070f5...
4,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,3,a4066f48ab8542d28660d03a5a06e578,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,4626d239-6145-4cc3-95ca-5640799dff03,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR1 • Pe...,2026-1,DEMO-C1,DEMO-CR1,P1,...,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1__ridge__2026...,models:/m-514702631c61416a900bc5a907c6b97a,ed906b992cc3d85b0641488fa01758037fd33efd7bd6f3...,22b3c1491ef5f01f1c646bc43fd8350d324bce87b3bf4e...,ridge,s3://pucpr-poc/lineage/poc/planilha-turmas/cat...,08e37bb2d3ce47f41a8a1415c0295e9f48b510b905bdf9...,RECORDED,False,{'files': [{'key': 'mlflow/5/models/m-51470263...
5,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,2,7beefbcd58d04056b1b1626db6b58edb,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,7101622e-40b2-48a2-a0fa-22de9d1b80e0,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR1 • Pe...,2026-1,DEMO-C1,DEMO-CR1,P1,...,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1__ridge__2026...,models:/m-4fea18d068194777a0becbd5f77d67a2,8e6043209a63edaadbdd46de27731ca0c7fbcf9555c8d9...,22b3c1491ef5f01f1c646bc43fd8350d324bce87b3bf4e...,ridge,s3://pucpr-poc/lineage/poc/planilha-turmas/cat...,b37cd33ab0ad13f5a4940866be8122b37c7051c9b3e6c7...,RECORDED,False,{'files': [{'key': 'mlflow/5/models/m-4fea18d0...
6,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,1,70d22910a6bb4d9e89bd56e2a332c7d5,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,59f2d8ec-65fc-44ae-aa94-46f82adaf2a7,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR1 • Pe...,2026-1,DEMO-C1,DEMO-CR1,P1,.

'/reports/v3/manual-session-dd99d9b0-2038-46cb-8994-ffbd4c37009a.json'

Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [4]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
if evidence.get("campaign",{}).get("status") in ("FAILED","PARTIAL","CANCELLED","PAUSED"):
    report_status="PARTIAL"
display(report(NOTEBOOK_ID,evidence,status=report_status,generation="v3",limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))


'/reports/v3/08_auditoria_lineage.json'